# 03 · Results and the next decision

**Decision:** Feature research is complete for the declared four-policy scope. The fixed route has now passed development validation: calibrate the seven-family model for familiar policies; retain the raw centroid for unseen policies. The final artifacts are fitted on development only. The frozen candidate has now passed all 12 protected acceptance checks on 43,509 rows. Offline inference now passes real-artifact parity, network-isolation, recovery and serving-budget checks. The model/data cards and release instructions describe the completed local product.

This short notebook is the employer review path: the measured improvement, the failed alternatives, the remaining limitation and the concrete release gates.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Historical references: 2,029 rows, two policies; expanded research is a separate cohort.")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


In [ ]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from build_release_report import display_boundary
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence
from jigsaw_rules.released import released_evidence
from jigsaw_rules.expanded import expanded_evidence
from jigsaw_rules.retrieval import retrieval_evidence
from jigsaw_rules.resolution import resolution_evidence
from jigsaw_rules.formatting import formatting_evidence
from jigsaw_rules.feature_decision import decision_evidence
from build_expanded_report import display_figure as display_expanded
from build_formatting_report import display_figure as display_formatting

expanded = expanded_evidence(root)
retrieval = retrieval_evidence(root)
resolution = resolution_evidence(root)
formatting = formatting_evidence(root)
decision = decision_evidence(root)
assert expanded is not None and retrieval is not None and resolution is not None
assert formatting is not None and decision is not None
controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
released = released_evidence(root)
assert released is not None
print("Verified expanded study:", expanded["metadata"]["run_id"])


## Protected confirmation: all 12 fixed checks pass
Predictions were frozen in published commit `530ad79929012e807cb42a5253f7b92b090682ec` before eligible target access on September 9, 2026. The primary six-policy macro AUC improves from **0.6801 to 0.7770**, a **+0.0969** gain with paired 95% interval **[0.0898, 0.1051]**. Familiar-policy AUC reaches **0.8276** and unseen-policy AUC **0.6757**; every policy improves.

Overall log loss falls from 0.6685 to 0.5121, and Brier from 0.2360 to 0.1739. This is one fixed post-competition comparison on **43,509 rows**, including two policies wholly excluded from development. The interval conditions on six observed policies. No candidate was retuned on these targets; this is not a Kaggle score. [Frozen protocol and recovery](../docs/CONFIRMATION.md).

In [ ]:
from jigsaw_rules.confirmation_report import confirmation_evidence
from build_protected_report import display_figure as display_protected
protected = confirmation_evidence(root)
assert protected is not None
print("Protected decision:", protected["results"]["status"])
display(pd.Series(protected["results"]["checks"], name="Passed preregistered check"))
display_protected(root, "policy_gains")

## What the expanded research measured
Thirty-four fixed trained configurations share seven purged splits. The feature study adds/removes major families, checks full versus screened representations, and tests training-only NB/SVD controls. Frozen semantic scores are untrained comparison baselines. The 43,576-row reserve remains outside model selection.

The frozen semantic centroid reaches **0.7042 transfer AUC**, versus **0.4728** for the matched lexical reference. Its log loss improves from 0.8126 to 0.6237. Most of the gain comes from avoiding reversed lexical ranking on illegal-activity promotion; legal and medical advice remain difficult. The all-transfer model reaches 0.7989 on familiar policies but only 0.5515 on transfer.

Policy-macro AUC is the primary competition-oriented metric. These are post-competition development results, not a Kaggle score.

In [ ]:
display_expanded(root, "comparison")
selected = {"rule_examples", "word_semantic_scalar", "semantic_scalar_only", "qwen_centroid", "all_transfer"}
rows = [r for r in expanded["results"] if r["model"] in selected]
rows += [r for r in formatting["results"] if r["model"] in {"asymmetric_centroid", "semantic_intent"}]
rows += [{"model": "centroid_intent_mean (rejected)", "protocol": "heldout_rule", "metrics": decision["fusion"]["metrics"]}]
display(metric_table(rows, heldout=True).sort_values("Rule macro AUC", ascending=False))

## What would count as a feature improvement?
The family-ablation figure in notebook `02` measures additions to the same screened-word control. The table below compares each representation with the same-fold rule/example reference. A higher point estimate with an interval crossing zero remains an uncertain development observation. Probability losses and policy-level failures are part of the decision. Simultaneous intervals apply within each preregistered study; combining rows below does not create one joint correction over all research.

In [ ]:
contrasts = pd.DataFrame(expanded["uncertainty"] + retrieval["uncertainty"])
compared = contrasts[(contrasts.protocol == "heldout_rule") & (contrasts.reference == "rule_examples")]
display(compared.sort_values("observed_delta", ascending=False).head(10)[["candidate", "observed_delta", "ci_lower", "ci_upper", "simultaneous_lower", "simultaneous_upper"]].round(4))

## Why the feature search stopped
The campaign covers 188,595–188,598 candidate columns per fold and 323 fixed fits. Training screens retain 9,281–9,660 columns across separate banks. The strongest transfer representation is simpler than those banks: a frozen support-centroid comparison. The last semantic variants fail the declared acceptance rule. A fixed average has the highest point AUC (0.7086), but its gain is uncertain and probability/policy regressions fail all five checks. The stopping decision retains 0.7042, rather than selecting the largest reported number.

The fixed route now has executed development evidence. Nine nested inner fits isolate calibration from outer validation labels; all three reconstructed familiar models reproduce their saved predictions within 1.2e-16. Familiar calibration passes all six gates; unseen calibration fails four and is discarded. One candidate and one lexical reference are fitted on all 11,135 development rows. The exact candidate/reference and acceptance protocol were committed before protected scoring; prediction hashes were published before eligible labels were interpreted.

The standalone inference notebook still uses the named lexical reference. The accepted benchmark artifact is now packaged for offline inference. The authored walkthrough below illustrates familiar and new policies; the model/data cards and release guide state its scope. The original-training-only Kaggle notebook remains separate. [Completed milestone specification](../docs/FINAL_MODEL_PLAN.md).

In [ ]:
print("Expanded study:", gate["expanded_development"])
print("Feature gate:", gate["status"])
print("Final training justified:", gate["final_training_authorized_by_evidence"])

## From features to the fitted candidate
The final familiar pipeline retains **9,263 of 181,958 columns** across seven selected families. It contains no target encodings, community metadata or raw embedding coordinates. Unknown policies bypass this learned pipeline.

Nested calibration improves familiar-policy log loss from **0.4761 to 0.4689**; the paired 97.5% interval for improvement is **[0.0037, 0.0109]**. AUC stays near 0.799. Calibrating the centroid damages transfer log loss from **0.6237 to 0.6976**, so unseen-policy probabilities retain their original mapping. This is a calibration result, separate from feature-engineering gains. [Protocol and full results](../docs/MODEL_VALIDATION.md).

In [ ]:
from jigsaw_rules.model_validation import validation_evidence
from build_model_report import display_figure as display_model
model_validation = validation_evidence(root)
assert model_validation is not None
display(metric_table(model_validation["results"]))
display_model(root, "calibration")
print("Final selected columns:", model_validation["audit"]["final_artifact"]["selected_features"])
print("Reserved targets accessed during development fitting:", model_validation["audit"]["confirmation_targets_accessed"])

## Use the accepted model: authored examples
The package reuses the accepted classifier and exact frozen encoder. On six target-free parity rows, fresh offline probabilities differ from the cloud predictions by at most 0.0000013; identical-vector predictions match exactly. No protected targets are reopened. The four examples below were authored for demonstration and are not another accuracy test.

The measured CPU run used four threads: 28.32 seconds for verification/loading plus the first six predictions, 1.11 seconds per comment for the four-example warm batch, and 3.88 GiB peak memory. These are small-batch measurements on the recorded host. Closest examples describe embedding similarity; they are not causal explanations. [Model card](../MODEL_CARD.md) · [Offline delivery and downloads](../docs/DELIVERY.md).

In [ ]:
from build_delivery_report import delivery_evidence
delivery = delivery_evidence(root)
with pd.option_context("display.max_colwidth", 100):
    display(pd.DataFrame(delivery["examples"])[["comment", "probability", "route", "nearest_example"]])
print("Offline checks passed:", all(delivery["checks"].values()))

### Try your own comment locally
Restore the model bundle using the delivery guide, then set `RUN_LOCAL_DEMO = True`. Edit the in-memory `demo_inputs` copy in the final cell to supply a comment, rule and two examples of each class. Keep the saved demo evidence unchanged. Missing required text is rejected. The default public review reads saved evidence only.

In [ ]:
RUN_LOCAL_DEMO = False
if RUN_LOCAL_DEMO:
    from jigsaw_rules.offline import OfflineModel
    demo_model = OfflineModel(root / delivery["bundle"], delivery["bundle_manifest_sha256"], root / "runs/demo_cache")
    demo_inputs = pd.read_json(root / "configs/demo.json")
    demo_predictions, demo_details = demo_model.predict(demo_inputs)
    display(demo_predictions)

## Explore the reasoning
[02 · Research methods, screening and ablations](02_baseline_and_review.ipynb) · [04 · Per-policy and probability diagnostics](04_semantic_benchmark.ipynb) · [01 · Leakage boundaries](01_data_and_validation.ipynb) · [Completed release milestones](../docs/ROADMAP.md).

The historical two-policy search is preserved in the [research record](../docs/FEATURE_RESEARCH.md); its scores are not compared numerically with this larger cohort as if only the features had changed.